# FX Pairs Dataset

Foreign exchange OHLCV data for G10 majors and crosses.

| Property | Value |
|----------|-------|
| **Provider** | OANDA |
| **Asset Class** | Currency |
| **Frequency** | Daily, 4-hourly |
| **Symbols** | 20 FX pairs |
| **Coverage** | 2011-2025 |
| **Size** | ~17 MB |
| **API Key** | `OANDA_API_KEY` (free) |
| **Loader** | `load_fx_pairs()` |

In [1]:
"""FX Pairs - download, explore, and update workflow."""

import json
import os
from pathlib import Path

import polars as pl
import yaml
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

True

## 1. Configuration

The FX dataset configuration defines which pairs to download, the date range,
and available frequencies. All parameters are stored in `config.yaml`.

In [2]:
# Load and display configuration
config_path = Path("config.yaml")
config = yaml.safe_load(config_path.read_text())

print("=== FX Configuration ===")
print(f"Provider: {config['fx']['provider']}")
print(f"Date range: {config['fx']['start']} to {config['fx']['end']}")
print(f"Frequencies: {config['fx']['frequencies']}")
print("\nPairs by category:")
for category, info in config["fx"]["pairs"].items():
    pairs = info["pairs"]
    print(f"  {category.capitalize()} ({len(pairs)}): {', '.join(pairs)}")

total_pairs = sum(len(info["pairs"]) for info in config["fx"]["pairs"].values())
print(f"\nTotal: {total_pairs} pairs")

=== FX Configuration ===
Provider: oanda
Date range: 2011-01-01 to 2025-12-31
Frequencies: ['daily', '4h']

Pairs by category:
  Majors (4): EURUSD, GBPUSD, USDJPY, USDCHF
  Commodity (3): AUDUSD, USDCAD, NZDUSD
  Crosses (13): EURGBP, EURJPY, EURCHF, EURCAD, EURAUD, GBPJPY, GBPCHF, GBPAUD, AUDJPY, CHFJPY, CADJPY, NZDJPY, AUDNZD

Total: 20 pairs


## 2. API Key Setup

OANDA provides free API access for historical FX data.

### Getting an OANDA API Key

1. Create a free practice account at [OANDA](https://www.oanda.com/)
2. Navigate to **Manage API Access** in your account settings
3. Generate a new API token
4. Add to your `.env` file in the repository root:

```bash
OANDA_API_KEY=your-api-key-here
```

The key format is typically: `xxxxxxxx-yyyyyyyy` (two parts separated by hyphen)

In [3]:
# Verify API key is configured
api_key = os.getenv("OANDA_API_KEY")
if api_key:
    # Show partial key for verification (first 8 chars)
    print(f"OANDA_API_KEY: {api_key[:8]}... (configured)")
else:
    print("WARNING: OANDA_API_KEY not set in environment")
    print("Add to .env file: OANDA_API_KEY=your-key-here")

OANDA_API_KEY: 808d9bb5... (configured)


## 3. Download Data

The download uses the `ml4t-data` library which handles:
- Rate limiting (OANDA allows 100 requests/second)
- Data validation
- Consistent schema output

**Note**: First-time download takes ~30 seconds per frequency (20 pairs each).

In [4]:
def download_fx_data(frequency: str = "4h", dry_run: bool = False):
    """Download FX data from OANDA.

    Args:
        frequency: "daily" or "4h"
        dry_run: If True, show what would be downloaded without doing it
    """
    from ml4t.data.providers.oanda import OandaProvider

    from utils import ML4T_DATA_PATH

    api_key = os.getenv("OANDA_API_KEY")
    if not api_key:
        raise ValueError("OANDA_API_KEY not set. See API Key Setup section.")

    # Load config
    config = yaml.safe_load(config_path.read_text())
    fx_config = config["fx"]

    # Flatten pairs list
    pairs = []
    for category_info in fx_config["pairs"].values():
        pairs.extend(category_info["pairs"])

    output_dir = ML4T_DATA_PATH / "fx" / "market"
    output_path = output_dir / f"{frequency}.parquet"

    print(f"=== FX Download ({frequency}) ===")
    print(f"Pairs: {len(pairs)}")
    print(f"Date range: {fx_config['start']} to {fx_config['end']}")
    print(f"Output: {output_path}")

    if dry_run:
        print("\n[DRY RUN] Would download:")
        for pair in pairs:
            print(f"  {pair}")
        return

    # Initialize provider
    provider = OandaProvider(api_key=api_key)

    # Download each pair
    all_data = []
    print(f"\nDownloading {len(pairs)} pairs...")
    for pair in pairs:
        print(f"  {pair}...", end=" ", flush=True)
        try:
            # OANDA uses format: EUR_USD (with underscore)
            oanda_pair = f"{pair[:3]}_{pair[3:]}"
            df = provider.fetch_ohlcv(oanda_pair, fx_config["start"], fx_config["end"], frequency)
            all_data.append(df)
            print(f"OK ({len(df):,} rows)")
        except Exception as e:
            print(f"ERROR: {e}")

    if not all_data:
        raise RuntimeError("No data downloaded!")

    # Combine and save
    output_dir.mkdir(parents=True, exist_ok=True)
    combined = pl.concat(all_data)
    combined.write_parquet(output_path)

    print("\n=== Complete ===")
    print(f"Total rows: {len(combined):,}")
    print(f"Pairs: {combined['symbol'].n_unique()}")
    print(f"Date range: {combined['timestamp'].min()} to {combined['timestamp'].max()}")
    print(f"Saved to: {output_path}")

    return combined

### Download Daily Data

In [5]:
# Uncomment to download daily data
# download_fx_data(frequency="daily")

### Download 4-Hourly Data

In [6]:
# Uncomment to download 4-hourly data
# download_fx_data(frequency="4h")

### Dry Run (Preview)

See what would be downloaded without actually downloading:

In [7]:
download_fx_data(frequency="4h", dry_run=True)

=== FX Download (4h) ===
Pairs: 20
Date range: 2011-01-01 to 2025-12-31
Output: data/fx/market/4h.parquet

[DRY RUN] Would download:
  EURUSD
  GBPUSD
  USDJPY
  USDCHF
  AUDUSD
  USDCAD
  NZDUSD
  EURGBP
  EURJPY
  EURCHF
  EURCAD
  EURAUD
  GBPJPY
  GBPCHF
  GBPAUD
  AUDJPY
  CHFJPY
  CADJPY
  NZDJPY
  AUDNZD


## 4. Load and Explore

Once downloaded, use the loader throughout the book:

In [8]:
from data import load_fx_pairs

# Load 4-hourly data (default)
df = load_fx_pairs()

print(f"Shape: {df.shape}")
print(f"Pairs: {df['symbol'].n_unique()}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Memory: {df.estimated_size('mb'):.1f} MB")

Shape: (478640, 7)
Pairs: 20
Date range: 2011-01-02 14:00:00 to 2025-12-31 18:00:00
Memory: 25.1 MB


In [9]:
# Schema
df.schema

Schema([('timestamp', Datetime(time_unit='ms', time_zone=None)),
        ('symbol', String),
        ('open', Float64),
        ('high', Float64),
        ('low', Float64),
        ('close', Float64),
        ('volume', Int64)])

In [10]:
# Preview
df.head(10)

timestamp,symbol,open,high,low,close,volume
datetime[ms],str,f64,f64,f64,f64,i64
2011-01-02 18:00:00,"""AUD_JPY""",83.0,83.332,82.8,82.875,986
2011-01-02 22:00:00,"""AUD_JPY""",82.877,83.015,82.698,82.796,4378
2011-01-03 02:00:00,"""AUD_JPY""",82.8,82.98,82.782,82.924,1934
2011-01-03 06:00:00,"""AUD_JPY""",82.926,83.03,82.826,83.03,3949
2011-01-03 10:00:00,"""AUD_JPY""",83.032,83.158,82.986,83.01,5296
2011-01-03 14:00:00,"""AUD_JPY""",83.008,83.27,82.992,83.218,8030
2011-01-03 18:00:00,"""AUD_JPY""",83.22,83.274,83.004,83.104,3695
2011-01-03 22:00:00,"""AUD_JPY""",83.102,83.148,82.9,82.972,6046
2011-01-04 02:00:00,"""AUD_JPY""",82.968,83.14,82.79,82.948,7164


In [11]:
# Available pairs
print("Available pairs:")
for i, pair in enumerate(sorted(df["symbol"].unique().to_list()), 1):
    print(f"  {i:2}. {pair}")

Available pairs:
   1. AUD_JPY
   2. AUD_NZD
   3. AUD_USD
   4. CAD_JPY
   5. CHF_JPY
   6. EUR_AUD
   7. EUR_CAD
   8. EUR_CHF
   9. EUR_GBP
  10. EUR_JPY
  11. EUR_USD
  12. GBP_AUD
  13. GBP_CHF
  14. GBP_JPY
  15. GBP_USD
  16. NZD_JPY
  17. NZD_USD
  18. USD_CAD
  19. USD_CHF
  20. USD_JPY


### Coverage by Pair

In [12]:
# Coverage and basic stats by pair
coverage = (
    df.group_by("symbol")
    .agg(
        pl.col("timestamp").min().alias("first_date"),
        pl.col("timestamp").max().alias("last_date"),
        pl.len().alias("n_bars"),
        pl.col("close").mean().alias("avg_price"),
    )
    .sort("symbol")
)
coverage

symbol,first_date,last_date,n_bars,avg_price
str,datetime[ms],datetime[ms],u32,f64
"""AUD_JPY""",2011-01-02 18:00:00,2025-12-31 18:00:00,23940,87.082469
"""AUD_NZD""",2011-01-02 18:00:00,2025-12-31 18:00:00,23922,1.118384
"""AUD_USD""",2011-01-02 18:00:00,2025-12-31 18:00:00,23941,0.789923
"""CAD_JPY""",2011-01-02 18:00:00,2025-12-31 18:00:00,23919,91.199853
"""CHF_JPY""",2011-01-02 18:00:00,2025-12-31 18:00:00,23919,122.492732
…,…,…,…,…
"""NZD_JPY""",2011-01-02 14:00:00,2025-12-31 18:00:00,23938,78.243043
"""NZD_USD""",2011-01-02 14:00:00,2025-12-31 18:00:00,23943,0.702921
"""USD_CAD""",2011-01-02 14:00:00,2025-12-31 18:00:00,23945,1.23786


### Volatility Analysis

In [13]:
# Annualized volatility by pair
fx_vol = (
    df.with_columns(pl.col("close").pct_change().over("symbol").alias("returns"))
    .group_by("symbol")
    .agg(
        (pl.col("returns").std() * (252 * 6) ** 0.5).alias("annual_vol"),  # 6 bars/day for 4h
    )
    .sort("annual_vol", descending=True)
)
fx_vol

symbol,annual_vol
str,f64
"""AUD_JPY""",0.119523
"""NZD_JPY""",0.118443
"""GBP_JPY""",0.10921
"""NZD_USD""",0.107603
"""CAD_JPY""",0.107173
…,…
"""EUR_CAD""",0.079489
"""EUR_GBP""",0.074007
"""USD_CAD""",0.073169


## 5. Data Profile

Profiles document the dataset structure, statistics, and quality metrics.
They are stored alongside the data files.

In [14]:
from utils import ML4T_DATA_PATH

# Check for existing profile
profile_path = ML4T_DATA_PATH / "fx" / "market" / "4h_profile.json"

if profile_path.exists():
    profile = json.loads(profile_path.read_text())
    print("=== FX 4h Profile ===")
    rows = profile.get("total_rows", profile.get("rows"))
    cols_field = profile.get("columns")
    n_cols = profile.get(
        "total_columns",
        len(cols_field) if isinstance(cols_field, list) else cols_field,
    )
    print(f"Rows: {rows:,}" if rows is not None else "Rows: unknown")
    print(f"Columns: {n_cols}")
    if isinstance(cols_field, list):
        print("\nSchema:")
        for c in cols_field:
            print(f"  {c['name']}: {c['dtype']}")
        ts = next((c for c in cols_field if c.get("name") == "timestamp"), None)
        if ts:
            print(f"\nDate range: {ts['min']}")
            print(f"         to {ts['max']}")
else:
    print(f"Profile not found at {profile_path}")
    print("Generate with: python generate_profiles.py --dataset fx_pairs_4h")

=== FX 4h Profile ===
Rows: 478,640
Columns: 7

Schema:
  timestamp: Datetime(time_unit='ms', time_zone=None)
  symbol: String
  open: Float64
  high: Float64
  low: Float64
  close: Float64
  volume: Int64

Date range: 2011-01-02 14:00:00
         to 2025-12-31 18:00:00


### Generate/Refresh Profile

To regenerate the profile after downloading new data:

```bash
python generate_profiles.py --dataset fx_pairs_4h --force
python generate_profiles.py --dataset fx_pairs_daily --force
```

## 6. Loader Options

The loader supports filtering by frequency, pairs, and date range:

In [15]:
# Daily frequency
daily = load_fx_pairs(frequency="daily")
print(f"Daily data: {daily.shape}")

Daily data: (77480, 7)


In [16]:
# Specific pairs
majors = load_fx_pairs(pairs=["EUR_USD", "GBP_USD", "USD_JPY"])
print(f"Majors only: {majors.shape}")

Majors only: (71825, 7)


In [17]:
# Date range
recent = load_fx_pairs(start_date="2024-01-01")
print(f"2024 onwards: {recent.shape}")

2024 onwards: (62283, 7)


In [18]:
# Combined filters
filtered = load_fx_pairs(
    frequency="daily", pairs=["EUR_USD", "GBP_USD"], start_date="2020-01-01", end_date="2023-12-31"
)
print(f"EUR/GBP daily 2020-2023: {filtered.shape}")

EUR/GBP daily 2020-2023: (2068, 7)


## 7. Documentation

### OANDA API
- [OANDA REST API Documentation](https://developer.oanda.com/rest-live-v20/introduction/)
- [Instrument List](https://developer.oanda.com/rest-live-v20/pricing-ep/)

### FX Market Conventions
- Pairs are quoted as BASE/QUOTE (e.g., EUR/USD = euros per dollar)
- Major pairs include USD; crosses exclude USD
- Standard lot = 100,000 units of base currency

### Data Quality Notes
- OANDA provides mid-prices (average of bid/ask)
- Volume represents OANDA's internal trading volume, not global FX volume
- Weekend gaps are normal (FX markets close Friday 5pm ET to Sunday 5pm ET)

## 8. Updating Data

To update with the latest data, re-run the download:

```python
# Update to latest available data
download_fx_data(frequency="4h")
download_fx_data(frequency="daily")
```

The `ml4t-data` library handles incremental updates automatically when the
end date in the config extends beyond existing data.

**Tip**: Update the `end` date in `config.yaml` before re-downloading
to extend the coverage period.

## Summary

| Item | Value |
|------|-------|
| Pairs | 20 (4 majors, 3 commodity, 13 crosses) |
| Frequencies | Daily, 4-hourly |
| Coverage | 2011-2025 |
| Provider | OANDA (free API key) |
| Config | `config.yaml` |
| Loader | `load_fx_pairs(frequency, pairs, start_date, end_date)` |
| Profile | `$ML4T_DATA_PATH/fx/{frequency}_profile.json` |